# 01 — Eval suite on vLLM

Runs the 100-question eval suite against vLLM and measures how the time for a
full pass changes with the number of concurrent requests.

- **Accuracy**: correct completions out of 100 for the chosen prompt version.
- **Concurrency sweep**: the same suite at 1, 4, 8, 16 and 32 concurrent
  requests. vLLM batches concurrent requests on the GPU (continuous batching),
  so a full pass gets faster as concurrency goes up.

Trade questions always run one at a time, because they change the database.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: vLLM + LMCache in their own virtualenv ---
# The server runs as a separate process, so it gets a separate environment.
# That keeps its packages from clashing with the ones Colab ships in the notebook.
VENV = "/content/venv-vllm"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python vllm lmcache
os.environ["SERVING_VENV"] = VENV

# Record the exact versions so the results can be reproduced.
!uv pip freeze --python {VENV}/bin/python | grep -iE "^(vllm|lmcache|torch|transformers)==" | tee results/versions_vllm.txt

In [ ]:
from serving.launch import start_server, stop_server

LARGE_MODEL = "Qwen/Qwen3-8B-AWQ"
SMALL_MODEL = "Qwen/Qwen3-0.6B"
PROMPT = "v1"    # prompt version from agent/prompts.py

## Large model: accuracy and concurrency sweep

In [ ]:
server = start_server("vllm", LARGE_MODEL, log_name="eval_vllm_large")

In [ ]:
!python -m eval.run_eval --name eval_vllm_8b_{PROMPT}_c16 --model {LARGE_MODEL} --prompt {PROMPT} --concurrency 16

In [ ]:
for c in [1, 4, 8, 32]:
    !python -m eval.run_eval --name eval_vllm_8b_{PROMPT}_c{c} --model {LARGE_MODEL} --prompt {PROMPT} --concurrency {c}

In [ ]:
stop_server(server)

## Small model: accuracy for comparison

In [ ]:
server = start_server("vllm", SMALL_MODEL, log_name="eval_vllm_small")
!python -m eval.run_eval --name eval_vllm_0.6b_{PROMPT}_c16 --model {SMALL_MODEL} --prompt {PROMPT} --concurrency 16
stop_server(server)

## Results

In [ ]:
import pandas as pd

summary = pd.read_csv("results/summary.csv")
summary[["name", "concurrency", "correct", "accuracy_pct", "multi_step_correct", "errors",
         "wall_s", "read_only_wall_s", "trade_wall_s", "completion_tokens_per_s"]]

In [ ]:
# --- Download the results ---
!zip -qr /content/results_01_eval_vllm.zip results
from google.colab import files
files.download("/content/results_01_eval_vllm.zip")